버스 정류소 정보 API

소스: https://data.seoul.go.kr/dataList/OA-15067/S/1/datasetView.do

In [11]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

API_KEY = "58765778426e616e33307571626a65"
SERVICE = "subwayStationMaster"
TYPE = "xml"

rows = []
start = 1
end = 1000

while True:
    url = f"http://openapi.seoul.go.kr:8088/{API_KEY}/{TYPE}/{SERVICE}/{start}/{end}/"
    print("요청:", url)

    response = requests.get(url)
    response.encoding = 'utf-8'

    if response.status_code != 200:
        raise Exception(f"API 호출 실패: {response.status_code}")

    soup = BeautifulSoup(response.text, "xml")

    rows_xml = soup.find_all("row")

    # 데이터 없음 → 종료
    if not rows_xml:
        print("더 이상 데이터 없음 → 종료")
        break

    # 데이터 추출
    for item in rows_xml:
        rows.append({
            "BLDN_NM": item.find("BLDN_NM").text if item.find("BLDN_NM") else None,
            "ROUTE": item.find("ROUTE").text if item.find("ROUTE") else None,
            "LAT": item.find("LAT").text if item.find("LAT") else None,
            "LOT": item.find("LOT").text if item.find("LOT") else None,
        })

    start += 1000
    end += 1000

df = pd.DataFrame(rows)
df.to_excel("서울시_지하철역_위치정보.xlsx", index=False)

print("🎉 완료! 총 개수:", len(df))


요청: http://openapi.seoul.go.kr:8088/58765778426e616e33307571626a65/xml/subwayStationMaster/1/1000/
요청: http://openapi.seoul.go.kr:8088/58765778426e616e33307571626a65/xml/subwayStationMaster/1001/2000/
더 이상 데이터 없음 → 종료
🎉 완료! 총 개수: 783


지하철역 정보 API

소스: https://data.seoul.go.kr/dataList/OA-21232/S/1/datasetView.do

In [10]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

# -------------------------------------
# 1. API KEY
# -------------------------------------
API_KEY = "58765778426e616e33307571626a65"   # ← 본인 API 키 입력
SERVICE = "busStopLocationXyInfo"
TYPE = "xml"

# -------------------------------------
# 2. 데이터 저장 리스트
# -------------------------------------
rows = []

start = 1
end = 1000

while True:
    url = f"http://openapi.seoul.go.kr:8088/{API_KEY}/{TYPE}/{SERVICE}/{start}/{end}/"
    print("요청:", url)

    response = requests.get(url)
    response.encoding = "utf-8"

    if response.status_code != 200:
        raise Exception(f"API 호출 실패: {response.status_code}")

    soup = BeautifulSoup(response.text, "xml")

    # XML 내부의 row 태그들을 찾음
    items = soup.find_all("row")

    # 데이터가 더 이상 없다면 종료
    if len(items) == 0:
        print("더 이상 가져올 데이터 없음. 종료.")
        break

    # 데이터 추출
    for item in items:
        rows.append({
            "STOPS_NM": item.find("STOPS_NM").text if item.find("STOPS_NM") else None,  # 정류소명
            "XCRD": item.find("XCRD").text if item.find("XCRD") else None,              # X좌표
            "YCRD": item.find("YCRD").text if item.find("YCRD") else None,              # Y좌표
        })

    # 다음 페이지로 이동
    start += 1000
    end += 1000

# -------------------------------------
# 3. 엑셀 파일로 저장
# -------------------------------------
df = pd.DataFrame(rows)
df.to_excel("서울시_버스정류소_위치정보.xlsx", index=False)

print("\n🎉 완료! '서울시_버스정류소_위치정보.xlsx' 생성됨!")
print("총 데이터 건수:", len(df))


요청: http://openapi.seoul.go.kr:8088/58765778426e616e33307571626a65/xml/busStopLocationXyInfo/1/1000/
요청: http://openapi.seoul.go.kr:8088/58765778426e616e33307571626a65/xml/busStopLocationXyInfo/1001/2000/
요청: http://openapi.seoul.go.kr:8088/58765778426e616e33307571626a65/xml/busStopLocationXyInfo/2001/3000/
요청: http://openapi.seoul.go.kr:8088/58765778426e616e33307571626a65/xml/busStopLocationXyInfo/3001/4000/
요청: http://openapi.seoul.go.kr:8088/58765778426e616e33307571626a65/xml/busStopLocationXyInfo/4001/5000/
요청: http://openapi.seoul.go.kr:8088/58765778426e616e33307571626a65/xml/busStopLocationXyInfo/5001/6000/
요청: http://openapi.seoul.go.kr:8088/58765778426e616e33307571626a65/xml/busStopLocationXyInfo/6001/7000/
요청: http://openapi.seoul.go.kr:8088/58765778426e616e33307571626a65/xml/busStopLocationXyInfo/7001/8000/
요청: http://openapi.seoul.go.kr:8088/58765778426e616e33307571626a65/xml/busStopLocationXyInfo/8001/9000/
요청: http://openapi.seoul.go.kr:8088/58765778426e616e33307571626a65/

컬럼명 변환

In [13]:
import pandas as pd
import os

# ===============================
# ⚙️ 설정
# ===============================
input_files = [
    "서울시_버스정류소_위치정보.xlsx",
    "서울시_지하철역_위치정보.xlsx",
]
output_folder = "standardized_excels"
os.makedirs(output_folder, exist_ok=True)

# ===============================
# 🧩 1️⃣ 컬럼 매핑 정의
# ===============================
column_map = {

    # 좌표
    "XCRD": "lon",
    "YCRD": "lat",
    "LOT": "lon",
    "LAT": "lat",

    # 혹시 영어 소문자 형태로 입력될 경우 대비
    "xcrd": "lon",
    "ycrd": "lat",
    "lot": "lon",
    "lat": "lat",
    "lon": "lon",
}

# ===============================
# 🧩 2️⃣ 컬럼명 표준화 함수
# ===============================
def standardize_columns(df):
    new_cols = []
    for col in df.columns:
        col_clean = col.strip().replace(" ", "").replace("_", "").upper()

        if col_clean in column_map:
            new_cols.append(column_map[col_clean])
        else:
            new_cols.append(col.lower())
    df.columns = new_cols
    return df


# ===============================
# 🧩 3️⃣ 파일 처리
# ===============================
for file in input_files:
    print(f"📄 {file} 처리 중...")

    df = pd.read_excel(file, engine="openpyxl")

    # 컬럼 표준화
    df = standardize_columns(df)

    # 결과 저장
    output_path = os.path.join(output_folder, os.path.basename(file))
    df.to_excel(output_path, index=False)

    print(f"✅ 완료: {output_path}")

print("\n🎯 모든 파일 컬럼명(lat, lon, place_name) 기준으로 성공적으로 표준화되었습니다!")


📄 서울시_버스정류소_위치정보.xlsx 처리 중...
✅ 완료: standardized_excels\서울시_버스정류소_위치정보.xlsx
📄 서울시_지하철역_위치정보.xlsx 처리 중...
✅ 완료: standardized_excels\서울시_지하철역_위치정보.xlsx

🎯 모든 파일 컬럼명(lat, lon, place_name) 기준으로 성공적으로 표준화되었습니다!


각 관광지별 500m 반경 내 지하철역, 버스정류소 개수 세어 엑셀로 저장

In [14]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, asin

# ============================================
# Haversine 거리 계산 함수 (미터 단위)
# ============================================
def haversine(lon1, lat1, lon2, lat2):
    R = 6371000
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    return R * c


# ============================================
# 1. 데이터 로드
# ============================================
places = pd.read_excel("merged_clean.xlsx")

# 관광지 좌표 컬럼명 정리
places = places.rename(columns={"longitude": "lon", "latitude": "lat"})

# standardized_excels 폴더 파일 로드
bus = pd.read_excel("standardized_excels/서울시_버스정류소_위치정보.xlsx")
subway = pd.read_excel("standardized_excels/서울시_지하철역_위치정보.xlsx")


# ============================================
# 2. 500m proximity 계산
# ============================================
result_rows = []

for idx, row in places.iterrows():
    pname = row["place_name"]
    plat = row["lat"]
    plon = row["lon"]

    # --- 지하철역 500m ---
    subway["dist"] = subway.apply(
        lambda x: haversine(plon, plat, x["lon"], x["lat"]), axis=1
    )
    subway_count = (subway["dist"] <= 500).sum()

    # --- 버스정류소 500m ---
    bus["dist"] = bus.apply(
        lambda x: haversine(plon, plat, x["lon"], x["lat"]), axis=1
    )
    bus_count = (bus["dist"] <= 500).sum()

    result_rows.append({
        "TourSpot": pname,
        "SubwayStations_500m": int(subway_count),
        "BusStops_500m": int(bus_count)
    })


# ============================================
# 3. 저장
# ============================================
result_df = pd.DataFrame(result_rows)
result_df.to_excel("tour_proximity_result.xlsx", index=False)

print("\n🎉 tour_proximity_result.xlsx 생성 완료!")



🎉 tour_proximity_result.xlsx 생성 완료!
